# 00 — Diagrams

Schematic figures explaining the QAOA algorithm and how the portfolio
problem maps onto it. Companion to `00_visuals.ipynb` (which carries the
data-driven motivational plots).

All figures saved as PDFs to `plots/visuals/` (Drive on Colab, local
repo otherwise). Colour palette comes from `palette/palette.json`.

> **Runtime:** ~5 s on Colab CPU. Pure matplotlib — no data is loaded.


### SETUP


In [ ]:
# === Bootstrap (Colab + local) ===
import os, urllib.request as _u
exec((open('../scripts/bootstrap.py') if os.path.exists('../scripts/bootstrap.py') else _u.urlopen('https://raw.githubusercontent.com/egil10/fys5419/main/project2/code/scripts/bootstrap.py')).read())

# === Project imports ===
import json
from itertools import product

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch, Rectangle

from scripts.colab import out_dir
from scripts.snp   import apply_style

PALETTE = json.loads(open('../palette/palette.json', encoding='utf-8').read())
apply_style()

PLOTS_DIR = out_dir('plots', 'visuals')
print(f'Plots will be saved to: {PLOTS_DIR}')


def heading(fig, bold, subtitle=None, x=0.04, y=0.98, gap=0.06):
    """Two-line heading in figure coords — robust to ax.axis('off')."""
    fig.text(x, y, bold, ha='left', va='top',
             fontsize=13, fontweight='bold',
             color=PALETTE['charcoal'])
    if subtitle:
        fig.text(x, y - gap, subtitle, ha='left', va='top',
                 fontsize=10, color=PALETTE['grey'])


### 1. QAOA CIRCUIT ANSATZ

The QAOA state alternates a cost-unitary $U_C(\gamma_\ell)=e^{-i\gamma_\ell H_C}$
(diagonal in the computational basis, red) and a mixer $U_M(\beta_\ell)=e^{-i\beta_\ell H_M}$
(off-diagonal X-rotations, blue) for $p$ layers, on top of the uniform
superposition $|+\rangle^{\otimes n}$. A classical optimiser (COBYLA) updates
$(\boldsymbol{\gamma}, \boldsymbol{\beta})$ from the measured expectation
$\langle H_C \rangle$ to lower the cost.


In [ ]:
n_q = 5

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(-0.8, 12.2)
ax.set_ylim(-2.4, n_q + 1.2)
ax.set_aspect('equal')
ax.axis('off')

x_init   = 0.0
x_H      = 0.9
x_blocks = [2.3, 3.7, 6.7, 8.1]
x_dots   = 5.2
x_meas   = 9.6
x_out    = 11.0
wire_end = x_meas + 0.4
y_top    = n_q - 1

# Quantum wires
for i in range(n_q):
    y = y_top - i
    ax.plot([x_init + 0.25, wire_end], [y, y],
            color=PALETTE['charcoal'], lw=1.0, zorder=1)
    ax.text(x_init - 0.05, y, r'$|0\rangle$',
            ha='right', va='center', fontsize=11,
            color=PALETTE['charcoal'])

# Hadamard column
for i in range(n_q):
    y = y_top - i
    ax.add_patch(Rectangle((x_H - 0.27, y - 0.27), 0.54, 0.54,
                           facecolor='white',
                           edgecolor=PALETTE['charcoal'],
                           lw=1.2, zorder=3))
    ax.text(x_H, y, 'H', ha='center', va='center',
            fontsize=10, fontweight='bold',
            color=PALETTE['charcoal'])

# Alternating cost / mixer blocks
block_w  = 0.95
y_bot    = -0.45
block_h  = (y_top - y_bot) + 0.45
block_specs = [
    (x_blocks[0], r'$U_C(\gamma_1)$', PALETTE['coral']),
    (x_blocks[1], r'$U_M(\beta_1)$',  PALETTE['sky']),
    (x_blocks[2], r'$U_C(\gamma_p)$', PALETTE['coral']),
    (x_blocks[3], r'$U_M(\beta_p)$',  PALETTE['sky']),
]
for x, lab, col in block_specs:
    ax.add_patch(FancyBboxPatch(
        (x - block_w / 2, y_bot), block_w, block_h,
        boxstyle='round,pad=0.02,rounding_size=0.10',
        facecolor=col, edgecolor=PALETTE['charcoal'],
        lw=1.2, alpha=0.85, zorder=2))
    ax.text(x, y_bot + block_h + 0.18, lab,
            ha='center', va='bottom', fontsize=11,
            color=PALETTE['charcoal'])

# Continuation dots between layers 1 and p
ax.text(x_dots, y_top / 2, r'$\cdots$',
        ha='center', va='center', fontsize=24,
        color=PALETTE['charcoal'])

# Per-wire measurement boxes (with arc + arrow glyph)
for i in range(n_q):
    y = y_top - i
    ax.add_patch(Rectangle((x_meas - 0.30, y - 0.27), 0.6, 0.54,
                           facecolor='white',
                           edgecolor=PALETTE['charcoal'],
                           lw=1.2, zorder=3))
    theta = np.linspace(np.pi, 2 * np.pi, 30)
    ax.plot(x_meas + 0.16 * np.cos(theta),
            y - 0.05 + 0.16 * np.sin(theta),
            color=PALETTE['charcoal'], lw=1.0, zorder=4)
    ax.annotate('', xy=(x_meas + 0.18, y + 0.14),
                xytext=(x_meas - 0.02, y - 0.05),
                arrowprops=dict(arrowstyle='-',
                                color=PALETTE['charcoal'],
                                lw=1.0), zorder=4)

# Bitstring output
ax.annotate('', xy=(x_out, y_top / 2),
            xytext=(x_meas + 0.35, y_top / 2),
            arrowprops=dict(arrowstyle='->',
                            color=PALETTE['charcoal'], lw=1.0))
ax.text(x_out + 0.15, y_top / 2,
        r'bitstring  $\boldsymbol{x}$',
        ha='left', va='center', fontsize=11,
        color=PALETTE['charcoal'])

# Column captions
ax.text(x_H, y_top + 0.95, 'prepare\n$|+\\rangle^{\\otimes n}$',
        ha='center', va='bottom', fontsize=9,
        color=PALETTE['grey'])
ax.text((x_blocks[0] + x_blocks[1]) / 2, y_top + 1.05,
        r'layer $\ell=1$',
        ha='center', va='bottom', fontsize=9,
        color=PALETTE['grey'])
ax.text((x_blocks[2] + x_blocks[3]) / 2, y_top + 1.05,
        r'layer $\ell=p$',
        ha='center', va='bottom', fontsize=9,
        color=PALETTE['grey'])
ax.text(x_meas, y_top + 0.95, 'measure',
        ha='center', va='bottom', fontsize=9,
        color=PALETTE['grey'])

# Classical outer loop feedback arrow
ax.add_patch(FancyArrowPatch(
    (x_meas, -1.55), (x_blocks[0], -1.55),
    arrowstyle='->', mutation_scale=15,
    color=PALETTE['red'], lw=1.5,
    connectionstyle='arc3,rad=-0.25'))
ax.text((x_blocks[0] + x_meas) / 2, -2.05,
        r'classical outer loop (COBYLA) updates '
        r'$(\boldsymbol{\gamma},\,\boldsymbol{\beta})$',
        ha='center', va='top',
        color=PALETTE['red'], fontsize=10)

heading(fig, 'QAOA ansatz',
        r'$|\psi(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle = '
        r'\prod_{\ell=1}^{p} e^{-i\beta_\ell H_M}\,'
        r'e^{-i\gamma_\ell H_C}\,'
        r'H^{\otimes n}|0\rangle^{\otimes n}$',
        y=0.98, gap=0.07)

fig.savefig(PLOTS_DIR / 'qaoa_circuit.pdf', bbox_inches='tight')
plt.show()


### 2. FROM PRICES TO BITSTRINGS

The full pipeline that turns market prices into a quantum optimisation
instance: log-returns + sample statistics produce $(\boldsymbol{\mu},\boldsymbol{\Sigma})$,
mean-variance + cardinality penalty give the QUBO cost $C(\boldsymbol{x})$,
the bit-to-spin map $x_i=(1-z_i)/2$ converts it to an Ising Hamiltonian
$H_C$, and QAOA prepares, optimises, and samples to return the optimal
bitstring $\boldsymbol{x}^\star$.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_xlim(0, 13)
ax.set_ylim(0, 5)
ax.set_aspect('equal')
ax.axis('off')

boxes = [
    ('Prices',           r'$P_i(t)$',
     PALETTE['ivory']),
    ('Returns + moments', r'$r_i(t),\ \boldsymbol{\mu},\,\boldsymbol{\Sigma}$',
     PALETTE['parchment']),
    ('Penalised QUBO',   r'$-\boldsymbol{\mu}^\top\boldsymbol{x}'
                          r'+\lambda\,\boldsymbol{x}^\top\boldsymbol{\Sigma}\boldsymbol{x}'
                          r'+A(\sum_i x_i-K)^2$',
     PALETTE['sky']),
    ('Ising $H_C$',      r'$\sum_i h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j$',
     PALETTE['coral']),
    ('QAOA + measure',   r'optimal bitstring $\boldsymbol{x}^\star$',
     PALETTE['ochre']),
]
arrow_labels = [
    'log-return\n+ sample stats',
    'mean-variance\n+ cardinality penalty',
    r'bit$\to$spin:' '\n' r'$x_i=(1-z_i)/2$',
    'prepare,\noptimise, sample',
]

n_box   = len(boxes)
margin  = 0.25
box_h   = 1.45
box_w   = 2.20
gap     = (13 - 2 * margin - n_box * box_w) / (n_box - 1)
y_box   = 2.0

for i, (label, eqn, col) in enumerate(boxes):
    x = margin + i * (box_w + gap)
    ax.add_patch(FancyBboxPatch(
        (x, y_box), box_w, box_h,
        boxstyle='round,pad=0.05,rounding_size=0.12',
        facecolor=col, edgecolor=PALETTE['charcoal'],
        lw=1.3, alpha=0.90, zorder=2))
    ax.text(x + box_w / 2, y_box + box_h - 0.30, label,
            ha='center', va='top', fontweight='bold', fontsize=10,
            color=PALETTE['charcoal'])
    ax.text(x + box_w / 2, y_box + box_h / 2 - 0.20, eqn,
            ha='center', va='center', fontsize=8.5,
            color=PALETTE['charcoal'])

    if i < n_box - 1:
        xs = x + box_w + 0.08
        xe = x + box_w + gap - 0.08
        ax.add_patch(FancyArrowPatch(
            (xs, y_box + box_h / 2), (xe, y_box + box_h / 2),
            arrowstyle='->', mutation_scale=14,
            color=PALETTE['charcoal'], lw=1.3))
        ax.text((xs + xe) / 2, y_box + box_h + 0.30,
                arrow_labels[i],
                ha='center', va='bottom', fontsize=8.5,
                color=PALETTE['grey'])

# Layer bands: data | problem | quantum
band_y, band_h = 1.0, 0.55
bands = [
    (0, 2, 'data layer',    PALETTE['warm_grey']),
    (2, 4, 'problem layer', PALETTE['blue_muted']),
    (4, 5, 'quantum layer', PALETTE['ochre']),
]
for i0, i1, name, col in bands:
    x0 = margin + i0 * (box_w + gap)
    x1 = margin + (i1 - 1) * (box_w + gap) + box_w
    ax.add_patch(FancyBboxPatch(
        (x0, band_y), x1 - x0, band_h,
        boxstyle='round,pad=0.0,rounding_size=0.08',
        facecolor='none', edgecolor=col, lw=1.1, ls='--'))
    ax.text((x0 + x1) / 2, band_y + band_h / 2, name,
            ha='center', va='center', fontsize=9,
            color=col, fontweight='bold')

heading(fig, 'From prices to bitstrings',
        r'pipeline: data $\to$ problem $\to$ quantum solver',
        y=0.98, gap=0.05)

fig.savefig(PLOTS_DIR / 'qaoa_pipeline.pdf', bbox_inches='tight')
plt.show()


### 3. X-MIXER vs XY-RING MIXER — SUBSPACE STRUCTURE

The mixer Hamiltonian determines which states the QAOA dynamics can reach.
The standard X-mixer $H_M=\sum_i X_i$ flips any single qubit, so it
connects every bitstring to its Hamming-weight neighbours — the dynamics
spans the full $2^n$ Hilbert space, and the cardinality constraint
$\sum_i x_i=K$ is enforced only by the penalty term in $H_C$.

The XY-ring mixer $H_M=\sum_i (X_i X_{i+1} + Y_i Y_{i+1})$ acts as adjacent
swaps; combined with the Dicke initial state $|D^n_K\rangle$ (uniform on
weight-$K$ bitstrings), the dynamics is **confined** to the weight-$K$
ring — every state QAOA can produce is automatically feasible.


In [ ]:
n_h = 4
K   = 2


def hamming_layout(n):
    bs = list(product([0, 1], repeat=n))
    by_w = {}
    for b in bs:
        by_w.setdefault(sum(b), []).append(b)
    pos = {}
    for w, group in by_w.items():
        m = len(group)
        r = 0.55 + 0.60 * w
        rot = (w % 2) * np.pi / max(m, 1)
        for j, b in enumerate(sorted(group)):
            theta = 2 * np.pi * j / m + rot + np.pi / 2
            pos[b] = (r * np.cos(theta), r * np.sin(theta))
    return pos, bs


pos, nodes = hamming_layout(n_h)
weight = lambda b: sum(b)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.subplots_adjust(top=0.78, bottom=0.13, left=0.04, right=0.98,
                    wspace=0.04)

for ax, panel in zip(axes, ['X', 'XY']):
    # Faint Hamming-weight rings, with k-labels nudged off the topmost vertex
    for w in range(n_h + 1):
        r = 0.55 + 0.60 * w
        ax.add_patch(Circle((0, 0), r, fill=False,
                            color=PALETTE['grid'],
                            lw=0.7, ls=':', zorder=1))
        theta_lab = np.deg2rad(118)
        ax.text(r * np.cos(theta_lab),
                r * np.sin(theta_lab) + 0.04,
                f'$k={w}$',
                ha='center', va='bottom', fontsize=7,
                color=PALETTE['grey'])

    if panel == 'X':
        edges = set()
        for b1 in nodes:
            for i in range(n_h):
                b2 = list(b1); b2[i] = 1 - b2[i]; b2 = tuple(b2)
                e = (b1, b2) if b1 < b2 else (b2, b1)
                edges.add(e)
        for b1, b2 in edges:
            x1, y1 = pos[b1]; x2, y2 = pos[b2]
            ax.plot([x1, x2], [y1, y2],
                    color=PALETTE['warm_grey'],
                    lw=0.55, alpha=0.60, zorder=2)
    else:
        edges = set()
        for b1 in nodes:
            if weight(b1) != K:
                continue
            for i in range(n_h):
                j = (i + 1) % n_h
                if b1[i] != b1[j]:
                    b2 = list(b1); b2[i], b2[j] = b2[j], b2[i]
                    b2 = tuple(b2)
                    if weight(b2) == K:
                        e = (b1, b2) if b1 < b2 else (b2, b1)
                        edges.add(e)
        for b1, b2 in edges:
            x1, y1 = pos[b1]; x2, y2 = pos[b2]
            ax.plot([x1, x2], [y1, y2],
                    color=PALETTE['red'],
                    lw=2.2, alpha=0.95, zorder=2)

    for b in nodes:
        x, y = pos[b]
        w = weight(b)
        if panel == 'XY' and w != K:
            ax.scatter([x], [y], s=120,
                       color=PALETTE['warm_grey'],
                       edgecolor=PALETTE['grid'],
                       lw=0.6, alpha=0.45, zorder=3)
            ax.text(x, y - 0.25, ''.join(map(str, b)),
                    ha='center', va='top', fontsize=7,
                    color=PALETTE['grid'])
        elif panel == 'XY' and w == K:
            ax.scatter([x], [y], s=170,
                       color=PALETTE['red'],
                       edgecolor=PALETTE['charcoal'],
                       lw=1.0, zorder=4)
            ax.text(x, y - 0.27, ''.join(map(str, b)),
                    ha='center', va='top', fontsize=8,
                    color=PALETTE['charcoal'],
                    fontweight='bold')
        else:
            ax.scatter([x], [y], s=140,
                       color=PALETTE['blue_muted'],
                       edgecolor=PALETTE['charcoal'],
                       lw=0.7, zorder=3)
            ax.text(x, y - 0.25, ''.join(map(str, b)),
                    ha='center', va='top', fontsize=7,
                    color=PALETTE['charcoal'])

    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.3, 3.7)
    ax.set_aspect('equal')
    ax.axis('off')

# Per-panel titles + captions in figure coords
fig.text(0.07, 0.82, 'X-mixer  —  single bit flips',
         ha='left', va='bottom', fontsize=11, fontweight='bold',
         color=PALETTE['charcoal'])
fig.text(0.07, 0.08,
         'arrows leak across all Hamming weights;\n'
         r'feasibility $\sum_i x_i = K$ is enforced only by the '
         r'penalty term in $H_C$',
         ha='left', va='bottom', fontsize=9,
         color=PALETTE['grey'])

fig.text(0.55, 0.82,
         r'XY-ring mixer  —  initialised at $|D^n_K\rangle$',
         ha='left', va='bottom', fontsize=11, fontweight='bold',
         color=PALETTE['charcoal'])
fig.text(0.55, 0.08,
         f'dynamics confined to the weight-$k={K}$ ring;\n'
         'feasibility preserved by construction',
         ha='left', va='bottom', fontsize=9,
         color=PALETTE['grey'])

heading(fig,
        'X-mixer vs XY-ring mixer  —  subspace structure',
        r'$n=4$ qubits; rings group bitstrings by Hamming weight '
        r'$k = \sum_i x_i$',
        y=0.96, gap=0.05)

fig.savefig(PLOTS_DIR / 'mixer_subspace.pdf', bbox_inches='tight')
plt.show()
